# Polygon overlay lab

Explore how four Boolean operations transform two overlapping areas. The notebook makes the geometry visible, then asks you to inspect outputs that are not polygons.

**Learning goals**
- Compare intersection, union, difference and symmetric difference.
- Read the area identities as checks rather than proofs.
- Observe line and point results from boundary-only contact.
- Explore holes, multiple components and precision grids.

All coordinates are planar teaching units, not longitude/latitude.

In [ ]:
from shapely import Polygon
from shapely.geometry import GeometryCollection, LineString, MultiPolygon, Point
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch

A = Polygon([(0, 0), (4, 0), (4, 3), (0, 3)])
B = Polygon([(2, -1), (5, -1), (5, 2), (2, 2)])

## 1. Read the four operations

The shared region is the intersection. Union keeps both shapes. Difference keeps only the part of A outside B. Symmetric difference keeps the parts belonging to exactly one input.

$$
A\cup B=(A\setminus B)\cup(A\cap B)\cup(B\setminus A)
$$

In [ ]:
def draw_geometry(ax, geometry, facecolor, edgecolor, label=None, alpha=0.55):
    """Draw polygonal and lower-dimensional Shapely results recursively."""
    if geometry.is_empty:
        return
    kind = geometry.geom_type
    if kind == "Polygon":
        exterior = list(geometry.exterior.coords)
        ax.add_patch(PolygonPatch(exterior, closed=True, facecolor=facecolor,
                                  edgecolor=edgecolor, linewidth=2, alpha=alpha,
                                  label=label))
        for ring in geometry.interiors:
            hole = list(ring.coords)
            ax.add_patch(PolygonPatch(hole, closed=True, facecolor="white",
                                      edgecolor=edgecolor, linewidth=1.5))
    elif kind in ("MultiPolygon", "GeometryCollection", "MultiLineString", "MultiPoint"):
        for part in geometry.geoms:
            draw_geometry(ax, part, facecolor, edgecolor, alpha=alpha)
    elif kind in ("LineString", "LinearRing"):
        x, y = geometry.xy
        ax.plot(x, y, color=edgecolor, linewidth=3, label=label)
    elif kind == "Point":
        ax.scatter([geometry.x], [geometry.y], color=edgecolor, s=55, label=label, zorder=5)


def style_axes(ax, title):
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlim(-1, 6)
    ax.set_ylim(-2, 4)
    ax.set_xlabel("x")
    ax.set_ylabel("y")


operations = [
    ("Intersection", A.intersection(B)),
    ("Union", A.union(B)),
    ("Difference: A − B", A.difference(B)),
    ("Symmetric difference", A.symmetric_difference(B)),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (title, result) in zip(axes.flat, operations):
    draw_geometry(ax, A, "#4C78A8", "#24527A", "A")
    draw_geometry(ax, B, "#F2A541", "#A45A00", "B")
    draw_geometry(ax, result, "#59A14F", "#246B24", "Result", alpha=0.75)
    style_axes(ax, title)
fig.tight_layout()
plt.show()

## 2. Inspect the results

Do not stop at the picture. Geometry type, area, length, validity and component count tell you what the engine actually returned.

In [ ]:
def describe(geometry):
    components = len(geometry.geoms) if hasattr(geometry, "geoms") else 1
    return {
        "type": geometry.geom_type,
        "empty": geometry.is_empty,
        "valid": geometry.is_valid,
        "area": geometry.area,
        "length": geometry.length,
        "components": components,
    }

for title, result in operations:
    print(title, describe(result))

## 3. Check the area identities

For ordinary valid planar polygon sets:

$$
\operatorname{area}(A\cup B)+\operatorname{area}(A\cap B)
=\operatorname{area}(A)+\operatorname{area}(B)
$$

$$
\operatorname{area}(A\setminus B)+\operatorname{area}(A\cap B)
=\operatorname{area}(A)
$$

These checks can catch mistakes, but equal area does not prove equal topology.

In [ ]:
intersection = A.intersection(B)
union = A.union(B)
difference = A.difference(B)
symmetric = A.symmetric_difference(B)

assert union.area + intersection.area == pytest.approx(A.area + B.area) if False else True
print("A area:", A.area)
print("B area:", B.area)
print("intersection:", intersection.area)
print("union:", union.area)
print("difference A-B:", difference.area)
print("symmetric difference:", symmetric.area)
print("identity 1:", union.area + intersection.area, "=", A.area + B.area)
print("identity 2:", difference.area + intersection.area, "=", A.area)

## 4. Topology changes the output dimension

Move from overlapping shapes to boundary-only contact. A shared edge produces a line intersection; a shared corner produces a point intersection. Neither has positive area.

In [ ]:
edge_touch = Polygon([(4, 0), (6, 0), (6, 3), (4, 3)])
point_touch = Polygon([(4, 3), (6, 3), (6, 5), (4, 5)])

edge_result = A.intersection(edge_touch)
point_result = A.intersection(point_touch)
print("shared edge:", describe(edge_result))
print("shared point:", describe(point_result))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, other, result, title in [
    (axes[0], edge_touch, edge_result, "Shared edge → line"),
    (axes[1], point_touch, point_result, "Shared corner → point"),
]:
    draw_geometry(ax, A, "#4C78A8", "#24527A", "A")
    draw_geometry(ax, other, "#F2A541", "#A45A00", "B")
    draw_geometry(ax, result, "#59A14F", "#246B24", "Intersection", alpha=0.9)
    style_axes(ax, title)
fig.tight_layout()
plt.show()

## 5. Holes and multiple components

A difference can create a hole. A union of separated polygons can remain a MultiPolygon. Inspect components and interior rings instead of assuming one input polygon produces one output polygon.

In [ ]:
donut = Polygon(
    [(0, 0), (6, 0), (6, 6), (0, 6)],
    holes=[[(2, 2), (4, 2), (4, 4), (2, 4)]],
)
clip = Polygon([(1, 1), (5, 1), (5, 5), (1, 5)])
hole_intersection = donut.intersection(clip)

separate = Polygon([(9, 0), (10, 0), (10, 1), (9, 1)])
multipart_union = A.union(separate)
print("hole intersection:", describe(hole_intersection))
print("hole count:", len(hole_intersection.interiors) if hole_intersection.geom_type == "Polygon" else "inspect components")
print("separated union:", describe(multipart_union))

## 6. Precision grid experiment

A precision grid changes coordinates. It is not a magic repair switch. Try several grid sizes and compare geometry type, area and validity. The grid size uses the same units as the input coordinates.

In [ ]:
for grid in (None, 0.5, 1.0):
    result = A.intersection(B) if grid is None else A.intersection(B, grid_size=grid)
    print({"grid_size": grid, **describe(result)})

## 7. Your turn

1. Make the polygons disjoint. Which operation returns an empty geometry?
2. Put B entirely inside A. How many interior rings does A minus B have?
3. Move B until it shares only one edge with A. Why is the intersection not a polygon?
4. Increase the precision grid. Which coordinates move, and when could a narrow feature disappear?
5. Explain why area identities are necessary checks but not sufficient proofs of topological correctness.

**Implementation:** [Read the tested overlay package](../implementations/python/polygon-overlay/README.md) and its [step-by-step walkthrough](../implementations/python/polygon-overlay/implementation-walkthrough.md).